# Assignment 1: POS Tagging using CRF

**Student Name:** Honey Aye

**Dataset:** myPOS Version 3.0

**Model:** Conditional Random Field (CRF)

**Objective:**
Develop a Part-of-Speech (POS) tagger using the myPOS dataset (version 3.0) and evaluate the model on the test dataset.

In [1]:
!pip install sklearn-crfsuite

In [2]:
import sklearn_crfsuite
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import metrics

In [3]:
!git clone https://github.com/ye-kyaw-thu/myPOS.git

fatal: destination path 'myPOS' already exists and is not an empty directory.


In [4]:
train_path = "myPOS/corpus-ver-3.0/corpus/train.mypos-ver3.txt"



In [5]:
def parse_sentence(line):
  sentence = []
  tokens = line.strip().split()
  for token in tokens:
    word, tag = token.rsplit("/", 1)
    sentence.append((word, tag))
  return sentence



In [6]:
def parse_corpus(file_path):
    sentences = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                sentence = parse_sentence(line)
                sentences.append(sentence)

    return sentences

train_sents = parse_corpus(train_path)

print("Number of sentences:", len(train_sents))
print(train_sents[0])

Number of sentences: 42196
[('၁၉၆၂', 'num'), ('ခုနှစ်', 'n'), ('ခန့်မှန်း', 'v'), ('သန်းခေါင်စာရင်း', 'n'), ('အရ', 'ppm'), ('လူဦးရေ', 'n'), ('၁၁၅၉၃၁', 'num'), ('ယောက်', 'part'), ('ရှိ', 'v'), ('သည်', 'ppm'), ('။', 'punc')]


In [7]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        "word": word,
        "length": len(word),
        "BOS": i == 0,
        "EOS": i == len(sent) - 1
    }

    if i > 0:
        features["previous_word"] = sent[i - 1][0]

    if i < len(sent) - 1:
        features["next_word"] = sent[i + 1][0]

    return features


In [8]:
def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [tag for word, tag in sent]

def sent2tokens(sent):
    return [word for word, tag in sent]

In [9]:
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]
print(len(X_train))
print(len(y_train))
print(X_train[0][:2])
print(y_train[0])

42196
42196
[{'word': '၁၉၆၂', 'length': 4, 'BOS': True, 'EOS': False, 'next_word': 'ခုနှစ်'}, {'word': 'ခုနှစ်', 'length': 6, 'BOS': False, 'EOS': False, 'previous_word': '၁၉၆၂', 'next_word': 'ခန့်မှန်း'}]
['num', 'n', 'v', 'n', 'ppm', 'n', 'num', 'part', 'v', 'ppm', 'punc']


In [10]:
crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=100)

In [11]:
test_path = "myPOS/corpus-ver-3.0/corpus/otest.1k.txt"

test_sents = parse_corpus(test_path)

print(len(test_sents))
print(test_sents[0])

1000
[('တစ်', 'tn'), ('ကိုက်', 'n'), ('ကို', 'ppm'), ('ဝမ်', 'n'), ('ခုနှစ်ထောင်', 'tn'), ('ပါ', 'part'), ('။', 'punc')]


In [12]:
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]
y_pred = crf.predict(X_test)

In [13]:
from sklearn_crfsuite import metrics

labels = list(crf.classes_)

print(metrics.flat_f1_score(
    y_test,
    y_pred,
    average="weighted",
    labels=labels
))

0.9517230384750445


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
print(
    metrics.flat_classification_report(
        y_test,
        y_pred,
        labels=labels,
        digits=4
    )
)

              precision    recall  f1-score   support

         num     0.9796    0.9290    0.9536       155
           n     0.9226    0.9661    0.9439      2567
           v     0.9436    0.9395    0.9415      1851
         ppm     0.9819    0.9767    0.9793      2060
        part     0.9618    0.9505    0.9561      3151
        punc     1.0000    1.0000    1.0000      1270
        conj     0.8788    0.9195    0.8987       410
         adj     0.8735    0.8296    0.8510       358
         adv     0.8862    0.8385    0.8617       260
        pron     0.9682    0.9601    0.9641       476
          tn     0.9781    0.9571    0.9675       140
          fw     0.9623    0.5862    0.7286        87
         int     0.9583    0.9200    0.9388        25
          sb     1.0000    1.0000    1.0000         3
         abb     1.0000    0.8333    0.9091        12
          v|     0.0000    0.0000    0.0000         0

    accuracy                         0.9520     12825
   macro avg     0.8934   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

In [15]:
test_words = sent2tokens(test_sents[0])

print(f"{'Word':<20} {'Actual':<10} {'Predicted':<10}")
print("-" * 45)

for word, actual, predicted in zip(
    test_words,
    y_test[0],
    y_pred[0]
):
    print(f"{word:<20} {actual:<10} {predicted:<10}")

Word                 Actual     Predicted 
---------------------------------------------
တစ်                  tn         tn        
ကိုက်                n          part      
ကို                  ppm        ppm       
ဝမ်                  n          n         
ခုနှစ်ထောင်          tn         num       
ပါ                   part       part      
။                    punc       punc      


# Results

- Accuracy: 0.9520
- Weighted F1-score: 0.9517

The CRF model achieved good performance on the myPOS test dataset. Most common POS tags were predicted correctly, while a few rare or ambiguous tags were misclassified.